# Вариант 37 — cifar10 / resnet18
Сахтай Е.Н. · CSE8842 · Ботайбеков А.О.

**Модельный эксперимент ещё не выполнен.** Оригинальная карточка включена ниже.

№37. CIFAR-10 + прунинг и квантизация ★
Суть: CIFAR-10, ResNet-18; фокус — accuracy vs latency vs размер (deployment).
Вопрос: что дешевле для deployment: прунинг или квантизация, и можно ли
совмещать? : fine-tune после прунинга — восстановление качества.

Что делать по стадиям: 1. Данные (ст. 0–1). Seed 1037. cifar10. EDA + contact
sheet. 2. Классика (ст. 2). Лист обработки. HOG + LinearSVC: --classical hog.
Deployment-эталон: размер/latency классики. 3. Аугментации (ст. 3). basic.
Абляция none vs basic. 4. Модель (ст. 4). resnet18 (обучить).
Params/память/размер файла. 5. Обучение (ст. 5). SGD lr 0.1 cosine, 30 эпох.
Кривые, resume. 6. Оценка (ст. 6). Полная + robustness; таблица Pareto
accuracy/latency/размер. 7. Grad-CAM (ст. 7). 5+5; объект vs фон. 8. Перенос (ст.
8). Прунинг 50% каналов + INT8-квантизация (по отдельности и вместе, блок
11 — отправные точки, реализация твоя); fine-tune после прунинга (бонус). 9.
Отчёт (ст. 9). Latency/размер до/после, cost, ответ на вопрос (что дешевле),
таблица классика/scratch/pruned/quant.

Команды:

python train.py --dataset cifar10 --model resnet18 --aug basic --optimizer sgd --lr 0.1 \
 --scheduler cosine --epochs 30 --seed 1037 --out_dir drive/out-var37
python evaluate.py --ckpt drive/out-var37/best.pt --dataset cifar10 --model resnet18 \
 --split val --seed 1037 --classical hog --processing --gradcam --robustness --latency --out_dir
drive/eval-var37

Артефакты: таблица Pareto (4 конфигурации), код прунинга/квантизации,
отчёт. Ловушка: прунинг без fine-tune резко роняет acc — замеряй и
восстановление после дообучения.



## Исходники проекта
Эта ячейка восстанавливает код из пакета. Работа выполняется в Colab, файлы результатов — на вашем Drive.

In [ ]:
import base64, gzip, json, os, sys
from pathlib import Path
PROJECT=Path('/content/cv39_project')
PROJECT.mkdir(exist_ok=True)
payload = '...'
files=json.loads(gzip.decompress(base64.b64decode(payload)))
for relative, content in files.items():
    p=PROJECT/relative
    p.parent.mkdir(parents=True,exist_ok=True)
    p.write_text(content,encoding='utf-8')
os.chdir(PROJECT)
sys.path.insert(0,str(PROJECT))
print('Source ready:',PROJECT)


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT='/content/drive/MyDrive/CV39_Sakhtay/runs'
DATA='/content/data'
Path(DATA).mkdir(exist_ok=True)


In [ ]:
import torch
from cv39.util import environment
print(environment())
assert torch.cuda.is_available(), 'Select T4 GPU in Runtime settings'


## Параметры
SMOKE=True проверяет только короткое обучение. Для выполнения задания используйте False. UC Merced, QuickDraw и UTKFace предварительно разместите по README.

In [ ]:
VARIANT=37
SMOKE=False
# DATA можно заменить на папку с подготовленными данными на Drive.


## Обучение, оценка и специальные эксперименты
После обрыва повторите эту же ячейку. Resume допускает только прежнюю конфигурацию.

In [ ]:
import subprocess
cmd=[sys.executable,'run.py','--variant',str(VARIANT),'--out',OUT,'--data',DATA]
if SMOKE: cmd+=['--smoke','--stage','train','--workers','0']
subprocess.run(cmd,check=True)

## Отчёт по полученным результатам

In [ ]:
from report import build_reports
from IPython.display import display, Markdown
build_reports(OUT)
folder=Path(OUT)/('smoke' if SMOKE else 'full')/f'variant_{VARIANT:02d}'
if (folder/'README.md').exists(): display(Markdown((folder/'README.md').read_text(encoding='utf-8')))
else: print('Smoke проверяет код; основной отчёт находится в runs/full')

## Ручной анализ
Проверьте изображения, классы, причины ошибок и ответ на индивидуальный вопрос. Создайте REPORT_NOTES.md рядом с результатами. Открывайте docs/COVERAGE.md: автоматический сценарий не заменяет бонусы и интерпретацию.

In [ ]:
notes=folder/'REPORT_NOTES.md'
if not notes.exists(): notes.write_text('Опишите реальные наблюдения и ответ на вопрос карточки.\n',encoding='utf-8')
print(notes)

## Финальный test — один раз после выбора по val
По умолчанию выключен. Проверьте, что выбран исходный классификатор, а не другая domain/rotation модель.

In [ ]:
FINAL_TEST=False
if FINAL_TEST and not SMOKE:
    subprocess.run([sys.executable,'run.py','--variant',str(VARIANT),'--stage','test','--out',OUT,'--data',DATA],check=True)

## Перед сдачей
Сохраните исполненный .ipynb через меню Colab, загрузите README/JSON/CSV/рисунки в GitHub. Веса и данные оставьте на Drive.